# Install Dependencies

In [1]:
# install standard libraries
#%pip install tensorflow

In [2]:
#%pip install opencv-python

# Importing libraries

In [3]:
#import standard libraries
import cv2, random , os , numpy as np , matplotlib.pyplot as plt

In [4]:
# import tensorflow dependencies - functional api 
from tensorflow.keras.models import Model # Model(inp = [inp image, verifi img],out=[0,1])
from tensorflow.keras.layers import Layer, Conv2D, Dense, MaxPooling2D, Input, Flatten 
# layer- custom layers, conv2d - convolution , dense - fully connected layer, max pooling - pull out layers and shrink 
#  input - define what were passing to model , flatten- flows data from layers

import tensorflow as tf


# CREATE FOLDERS

In [5]:
# create 3 folders(paths) : anchor(input) / positive(verification image) / negative(verification image)
#setup paths 

pos_path = os.path.join('data','positive') # data/positive
neg_path = os.path.join('data','negative') #data/negative
anchor_path = os.path.join('data','anchor') #data/anchor


In [6]:
# make directories 
'''
os.makedirs(pos_path)
os.makedirs(neg_path)
os.makedirs(anchor_path)
'''

'\nos.makedirs(pos_path)\nos.makedirs(neg_path)\nos.makedirs(anchor_path)\n'

## Collect positive and Anchors

In [7]:
# import uuuid(universally unique identifier) to generate unique image names 
import uuid 

In [8]:
# uuid1, uuid2, uuid3, uuid4, uuid5  - formats you can use  
uuid.uuid1()

UUID('ad3f15f8-39a8-11f1-9ca1-40490f13fa1a')

In [9]:
'{}.jpg'.format(uuid.uuid1())

'ad542248-39a8-11f1-b271-40490f13fa1a.jpg'

In [10]:
os.path.join(anchor_path,'{}.jpg'.format(uuid.uuid1())) 

'data\\anchor\\ad62c8f3-39a8-11f1-a0fd-40490f13fa1a.jpg'

In [11]:
# connect to webcam 
import cv2
cap = cv2.VideoCapture(0)
while cap.isOpened(): # loop through every frame in webcam 
    ret , frame = cap.read() # returns val and actual frame

    frame = frame[120:120+250,200:200+250,:] # cut down frame to 250px - 250px 

    # collect anchors 
    if cv2.waitKey(1) & 0XFF == ord('a'):
        # save frame to respective folders using uuid 
        imgname = os.path.join(anchor_path,'{}.jpg'.format(uuid.uuid1())) # unique faile path 
        cv2.imwrite(imgname,frame)


    # collect positives
    if cv2.waitKey(1) & 0XFF == ord('p'):
        # save frame to respective folders using uuid 
        imgname = os.path.join(pos_path,'{}.jpg'.format(uuid.uuid1())) # unique faile path 
        cv2.imwrite(imgname,frame)



    # show image back to screen    
    cv2.imshow('Image Collection', frame) 
    
    # press q to break webcam connection
    if cv2.waitKey(1) & 0XFF == ord('q'):
        break


#release the webcam 
cap.release()
# close the frame showing image 
cv2.destroyAllWindows()


In [12]:
# click a - anchor , p- positive 
#plt.imshow(frame)
frame.shape # 250px x 250px x 3 channels

(250, 250, 3)

In [13]:
# slicing to get right shape [yaxis,xaixs]
#plt.imshow(frame[120:120+250,200:200+250,:])

# LOAD AND PREPROCESS IMAGES 

In [14]:
# get image directories 
# 3 vars = grab all images in the specific folder 
anchor = tf.data.Dataset.list_files(anchor_path+'\*.jpg').take(90)# get every file with \xxxxxx.jpg only 90 images
positive = tf.data.Dataset.list_files(pos_path+'\*.jpg').take(90)# number of samples should be same it only loads path not actual images
negative = tf.data.Dataset.list_files(neg_path+'\*.jpg').take(90)

In [15]:
anchor_path+'\*.jpg'

'data\\anchor\\*.jpg'

In [16]:
dir_test = anchor.as_numpy_iterator()
dir_test.next() # get next file( image id )

b'data\\anchor\\ae660ee4-381f-11f1-9094-40490f13fa1a.jpg'

## PREPROCESSING - SCALE AND RESIZE 
fxn to load image and resize it  and convert image val to 0-255 to 0-1

In [17]:
def preprocess(file_path):
    byte_img = tf.io.read_file(file_path)  # read img using file path treeated using bytes like obj 
    img = tf.io.decode_jpeg(byte_img)          # decode jpeg -> load img 
    
    img = tf.image.resize(img,(100,100))    # resize img -> preprocessing (100px, 100px, 3 channels) as per paper 
    img = img / 255.0              # divide it by 255 so it performs scaling and it returnns the image 0-1
    return img

In [18]:
img = preprocess('data\\anchor\\41833586-381f-11f1-80e5-40490f13fa1a.jpg')

In [19]:
img.shape

TensorShape([100, 100, 3])

In [20]:
img.numpy().min()

np.float32(0.014705882)

In [21]:
img.numpy().max()

np.float32(0.89730394)

In [22]:
#plt.imshow(img)

## CREATE LABELLED DATASET 

In [23]:
# use 
tf.ones_like(1)

<tf.Tensor: shape=(), dtype=int32, numpy=1>

In [24]:
tf.ones_like([1,1,3,4,555.5,666.0]) # - > set of ones not different vals 

<tf.Tensor: shape=(6,), dtype=float32, numpy=array([1., 1., 1., 1., 1., 1.], dtype=float32)>

In [25]:
# pass anchot + pos img as input // otp -> array os ones 
# same for negatives -> otp 0 

In [26]:
tf.ones(len(anchor))

<tf.Tensor: shape=(90,), dtype=float32, numpy=
array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
       1., 1., 1., 1., 1.], dtype=float32)>

In [27]:
tf.zeros(len(anchor))

<tf.Tensor: shape=(90,), dtype=float32, numpy=
array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.], dtype=float32)>

In [28]:
tf.data.Dataset.from_tensor_slices(tf.zeros(len(anchor))) # putting data into data loader 

<_TensorSliceDataset element_spec=TensorSpec(shape=(), dtype=tf.float32, name=None)>

In [29]:
# anc +pos -> 1
positives = tf.data.Dataset.zip((anchor, positive, tf.data.Dataset.from_tensor_slices(tf.ones(len(anchor)))))
# anc + neg -> 0
negatives = tf.data.Dataset.zip((anchor, negative, tf.data.Dataset.from_tensor_slices(tf.zeros(len(anchor)))))
data = positives.concatenate(negatives)

In [30]:
data

<_ConcatenateDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.float32, name=None))>

In [31]:
samples  = data.as_numpy_iterator()

In [32]:
samples.next() # keep iterating to dpoints

(b'data\\anchor\\d0316459-381f-11f1-9c26-40490f13fa1a.jpg',
 b'data\\positive\\6a3684f6-3820-11f1-ba5c-40490f13fa1a.jpg',
 np.float32(1.0))

## BUILD TRAIN AND TEST PARTITION

In [33]:
'''(b'data\\anchor\\85dbc779-381f-11f1-bc00-40490f13fa1a.jpg', # input 
 b'data\\positive\\a0df0918-3820-11f1-988b-40490f13fa1a.jpg',  # valid 
 np.float32(1.0))'''
def preprocess_twin(input_image, validation_image, label):
    return (preprocess(input_image), preprocess(validation_image), label)

In [34]:
res = preprocess_twin(b'data\\anchor\\85dbc779-381f-11f1-bc00-40490f13fa1a.jpg',
 b'data\\positive\\a0df0918-3820-11f1-988b-40490f13fa1a.jpg',
 np.float32(1.0))

In [35]:
res[0] # converts 100 , 100 , 3

<tf.Tensor: shape=(100, 100, 3), dtype=float32, numpy=
array([[[0.76985294, 0.7776961 , 0.75808823],
        [0.7590686 , 0.76691175, 0.7473039 ],
        [0.76593137, 0.782598  , 0.75710785],
        ...,
        [0.8622549 , 0.89362746, 0.8504902 ],
        [0.8784314 , 0.9098039 , 0.8607843 ],
        [0.8627451 , 0.8862745 , 0.83137256]],

       [[0.7683824 , 0.7713235 , 0.73995095],
        [0.7588235 , 0.76862746, 0.74019605],
        [0.76862746, 0.77843136, 0.75      ],
        ...,
        [0.8757353 , 0.90661764, 0.85784316],
        [0.8620098 , 0.8870098 , 0.84117645],
        [0.8610294 , 0.88529414, 0.82892156]],

       [[0.76715684, 0.76715684, 0.72205883],
        [0.7767157 , 0.7769608 , 0.73382354],
        [0.7772059 , 0.77818626, 0.7409314 ],
        ...,
        [0.8661765 , 0.8904412 , 0.8512255 ],
        [0.8691176 , 0.8926471 , 0.85343134],
        [0.8762255 , 0.89093137, 0.84632355]],

       ...,

       [[0.73995095, 0.7352941 , 0.7169118 ],
        [0.73

In [36]:
res[2]

np.float32(1.0)

In [37]:
#plt.imshow(res[0])

In [38]:
# build data loader pipeline 
data = data.map(preprocess_twin)
data = data.cache() # save processed data to ram 
data = data.shuffle(buffer_size = 1024) # pick 1024 dpoints 

In [39]:
sample = data.as_numpy_iterator()

In [40]:
eg = sample.next()

In [41]:
#plt.imshow(eg[0])

In [42]:
#plt.imshow(eg[1])

In [43]:
eg[2]

np.float32(1.0)

In [44]:
len(data)

180

In [45]:
round((70/100)*180)

126

In [46]:
# train  partition 
train_data = data.take(round(len(data)*.7))  # 70% training data 
train_data = train_data.batch(8) # pass through data as batch of 16
train_data = train_data.prefetch(4) # starts preprocessing next 4 images 

In [47]:
ts = train_data.as_numpy_iterator()

In [48]:
len(ts.next()[0]) # 8 images 

8

In [49]:
data

<_ShuffleDataset element_spec=(TensorSpec(shape=(100, 100, None), dtype=tf.float32, name=None), TensorSpec(shape=(100, 100, None), dtype=tf.float32, name=None), TensorSpec(shape=(), dtype=tf.float32, name=None))>

In [50]:
round(len(data)*.3)

54

In [51]:
# testing partition 
test_data = data.skip(round(len(data)*.7)) # skip first 70 % 
test_data = test_data.take(round(len(data)*.3)) # take only last 30 %
test_data = test_data.batch(8)
test_data = test_data.prefetch(4)

# Model Engineering 

### BUILDING EMBEDDING LAYER 

In [106]:
inp = Input(shape=(100,100,3), name="input_image") # using 3 because we have colored img in paper 105,105,1 
inp # none-> batch size 

<KerasTensor shape=(None, 100, 100, 3), dtype=float32, sparse=False, ragged=False, name=input_image>

In [107]:
c1 = Conv2D(64,(10,10), activation = 'relu')(inp)

In [108]:
c1 # shape = 91, 91 , 64 channels  bec of 100, 100 , 3 

<KerasTensor shape=(None, 91, 91, 64), dtype=float32, sparse=False, ragged=False, name=keras_tensor_27>

In [109]:
m1 = MaxPooling2D(64,(2,2), padding = 'same')(c1)
m1

<KerasTensor shape=(None, 46, 46, 64), dtype=float32, sparse=False, ragged=False, name=keras_tensor_28>

In [110]:
# second block 
c2 = Conv2D(128,(7,7), activation='relu')(m1)
m2 = MaxPooling2D(64,(2,2), padding = 'same')(c2)

In [111]:
c2 # check on the paper 

<KerasTensor shape=(None, 40, 40, 128), dtype=float32, sparse=False, ragged=False, name=keras_tensor_29>

In [112]:
m2

<KerasTensor shape=(None, 20, 20, 128), dtype=float32, sparse=False, ragged=False, name=keras_tensor_30>

In [113]:
# third block 
c3 = Conv2D(128,(4,4), activation='relu')(m2)
m3 = MaxPooling2D(64,(2,2), padding = 'same')(c3)

In [114]:
print(c3)
print(m3)

<KerasTensor shape=(None, 17, 17, 128), dtype=float32, sparse=False, ragged=False, name=keras_tensor_31>
<KerasTensor shape=(None, 9, 9, 128), dtype=float32, sparse=False, ragged=False, name=keras_tensor_32>


In [115]:
c4 = Conv2D(256,(4,4), activation = 'relu')(m3)

In [116]:
c4

<KerasTensor shape=(None, 6, 6, 256), dtype=float32, sparse=False, ragged=False, name=keras_tensor_33>

In [117]:
6*6*256

9216

In [118]:
f1 = Flatten()(c4)
f1

<KerasTensor shape=(None, 9216), dtype=float32, sparse=False, ragged=False, name=keras_tensor_34>

In [119]:
d1 = Dense(4096, activation = 'sigmoid')(f1)
d1

<KerasTensor shape=(None, 4096), dtype=float32, sparse=False, ragged=False, name=keras_tensor_35>

In [120]:
mod = Model(inputs=[inp], outputs=[d1], name='embedding')
mod.summary()

Model: "embedding"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)             │ (None, 100, 100, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_15 (Conv2D)                   │ (None, 91, 91, 64)          │          19,264 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_9 (MaxPooling2D)       │ (None, 46, 46, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_16 (Conv2D)                   │ (None, 40, 40, 128)         │         401,536 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_10 (MaxPooling2D)      │ (None, 20, 20, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_17 (Conv2D)                   │ (None, 17, 17, 128)         │         262,272 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_11 (MaxPooling2D)      │ (None, 9, 9, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_18 (Conv2D)                   │ (None, 6, 6, 256)           │         524,544 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_2 (Flatten)                  │ (None, 9216)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 4096)                │      37,752,832 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 38,960,448 (148.62 MB)

 Trainable params: 38,960,448 (148.62 MB)

 Non-trainable params: 0 (0.00 B)

In [123]:
def make_embedding():
    # create input layer 
    inp = Input(shape=(100,100,3), name="input_image")

# first block     
    # convolution layer + relu activation : convlay 2 imp things (64 = no of filters , 10px by 10px = filter shape )
    c1 = Conv2D(64,(10,10), activation = 'relu')(inp)

    # max pooling layer 
    m1 = MaxPooling2D(64,(2,2), padding = 'same')(c1)

    # inpu -> con+relu -> maxpool => conv + relu -> maxpool => conv +relu -> maxpool => coonv +relu -> fully connected + sigmoid l1 siamese distance 

 # second block 
    c2 = Conv2D(128,(7,7), activation='relu')(m1)
    m2 = MaxPooling2D(64,(2,2), padding = 'same')(c2)
    
# third block 
    c3 = Conv2D(128,(4,4), activation='relu')(m2)
    m3 = MaxPooling2D(64,(2,2), padding = 'same')(c3)

# final embedding block 
    c4 = Conv2D(256,(4,4), activation = 'relu')(m3)
    f1 = Flatten()(c4)
    d1 = Dense(4096, activation = 'sigmoid')(f1)



    # return embedding model imported using tensorflow.keras.models import Model
    return Model(inputs=[inp], outputs=[d1], name='embedding')

In [124]:
# build model 
model = make_embedding()

In [125]:
model.summary()

Model: "embedding"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)             │ (None, 100, 100, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_19 (Conv2D)                   │ (None, 91, 91, 64)          │          19,264 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_12 (MaxPooling2D)      │ (None, 46, 46, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_20 (Conv2D)                   │ (None, 40, 40, 128)         │         401,536 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_13 (MaxPooling2D)      │ (None, 20, 20, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_21 (Conv2D)                   │ (None, 17, 17, 128)         │         262,272 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_14 (MaxPooling2D)      │ (None, 9, 9, 128)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_22 (Conv2D)                   │ (None, 6, 6, 256)           │         524,544 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_3 (Flatten)                  │ (None, 9216)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 4096)                │      37,752,832 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 38,960,448 (148.62 MB)

 Trainable params: 38,960,448 (148.62 MB)

 Non-trainable params: 0 (0.00 B)